# DATA 266 Lab 2 - Part 1: Multimodal RAG over a read-only knowledge base
Modular pipeline in `rag/` (ingest -> figure descriptions -> hybrid index -> grounded answers). Run all cells top to bottom; `submission.csv` is written by Cell 7.

In [1]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
try:
    commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
except Exception:
    commit = ""
print({"commit": commit or "n/a (not a git checkout)", "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

{'commit': 'n/a (not a git checkout)', 'python': '3.13.16', 'torch': '2.11.0+cu130', 'gpu': 'NVIDIA A100-SXM4-40GB', 'seed': 266}


In [2]:
# Cell 2 - install dependencies (Colab/Kaggle). torch/numpy/pandas are already present.
# Versions are pinned in requirements.txt; on Kaggle, turn "Internet" ON for the install, or attach the packages as a dataset.
!pip -q install transformers==4.49.0 accelerate==1.3.0 bitsandbytes==0.45.2 sentence-transformers==3.4.1 chromadb==0.6.3 pdfplumber==0.11.5

In [3]:
import glob, os, pathlib, sys, zipfile

def find_repo_root():
    roots = [".", "/content", "/kaggle/working", "/kaggle/input"]
    for r in roots:
        for p in glob.glob(f"{r}/**/rag/config.py", recursive=True):
            return str(pathlib.Path(p).parents[1])
    dest = "/content/_repo" if os.path.isdir("/content") else "/kaggle/working/_repo"
    for r in roots:
        for z in glob.glob(f"{r}/**/*.zip", recursive=True):
            try:
                with zipfile.ZipFile(z) as zf:
                    if any(n.endswith("rag/config.py") for n in zf.namelist()):
                        zf.extractall(dest)
                        for p in glob.glob(f"{dest}/**/rag/config.py", recursive=True):
                            return str(pathlib.Path(p).parents[1])
            except zipfile.BadZipFile:
                pass
    return None

REPO = find_repo_root()
if REPO is None:
    raise SystemExit("rag/ not found - check that the zip/folder is attached under Add Data")
sys.path.insert(0, REPO)
print("using repo at:", REPO)

using repo at: _repo/data266-lab2-team


In [4]:
import os
os.environ["VLM_NAME"] = "Qwen/Qwen2.5-VL-3B-Instruct"
os.environ["LOAD_4BIT"] = "0"

In [5]:
# Cell 4 - load the vision-language model (reads figures AND writes answers). 4-bit Qwen2.5-VL-7B fits a 16 GB T4.
import logging; logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
from rag import config
from rag.llm import load_backend
backend = load_backend()
print("backend:", backend.name)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.53G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

backend: Qwen/Qwen2.5-VL-3B-Instruct


In [7]:
import os
os.environ["DATA_DIR"] = "/content/_repo/data266-lab2-team/data"
print(os.listdir(os.environ["DATA_DIR"]))   # should list kb_manifest.json and the 3 PDFs

['Document2_Supplement_A_Outturns_2011-2013.pdf', 'validate_submission.py', 'Document3_Briefing_B_Labour_Prices_Policy.pdf', 'Document.pdf', 'kb_manifest.json', 'Questions.csv', 'sample_submission.csv']


In [8]:
# Cell 5 - Task 1.1 ingestion + indexing: text chunks, tables, figure crops (with labels), figure descriptions,
# then a ChromaDB dense index + BM25 index. Low-trust material (DOC3 Appendix C) is stored but never retrieved.
from rag.pipeline import build_system
system = build_system(backend)
system.stats

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


{'items': 291,
 'by_doc_kind': {'DOC1/text': 224,
  'DOC2/text': 17,
  'DOC3/text': 14,
  'DOC1/figure': 13,
  'DOC3/figure': 7,
  'DOC2/figure': 7,
  'DOC1/table': 4,
  'DOC2/table': 3,
  'DOC3/table': 2},
 'figure_descriptions': {'figures': 26,
  'described': 26,
  'failed': 0,
  'seconds': 464.0},
 'dense_enabled': True,
 'build_seconds': 486.2}

In [9]:
# Cell 6 - smoke test on the first question. Never fatal: problems are shown, not raised.
from rag.pipeline import read_questions
from rag.memory import ConversationMemory
from rag.retrieve import retrieve
from rag.answer import answer_question
import json
q = read_questions()
first = q.iloc[0]
ev = retrieve(system.index, first["question"], ConversationMemory("smoke"))
ans, dbg = answer_question(first["question"], ev, ConversationMemory("smoke"), backend)
print(json.dumps(ans, indent=1)); print("issues:", dbg["issues"]); print("evidence:", [i.item_id for i in ev.items])

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


{
 "answer": "The global economic crisis around 2008 was sparked by the housing bubble in the United States, leading to a collapse in stock prices and a decline in output.",
 "status": "answered",
 "citations": [
  "DOC1:p2",
  "DOC1:p5"
 ],
 "figure_references": []
}
issues: []
evidence: ['DOC1-p2-c1', 'DOC1-p12-c3', 'DOC1-p2-c3', 'DOC1-p12-c4', 'DOC1-p24-c4', 'DOC1-p5-c0']


In [10]:
# Cell 7 - answer every question (conversation by conversation) and write submission.csv with no manual step.
# QUESTIONS_PATH is honoured so graders can swap in the hidden set.
import time, pathlib
from rag.pipeline import run_questions, validate_submission
from rag.config import questions_path, find_data_dir, work_dir
q = read_questions(questions_path(find_data_dir()))
t0 = time.time()
sub = run_questions(system, q, "submission.csv", work_dir() / "trace_p1.jsonl")
print(f"answered {len(sub)} questions in {time.time()-t0:.0f}s")
problems = validate_submission(sub, q)
print("format problems:", problems or "none")
sub.head(10)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/config

answered 40 questions in 127s
format problems: none


,id,conversation_id,question_id,answer,figure_references,citations,status
0,1,C001,Q001,The global economic crisis around 2008 was spa...,NONE,"DOC1:p2,DOC1:p5",answered
1,2,C002,Q002,Unemployment rates should be of concern becaus...,NONE,"DOC1:p24,DOC1:p30",answered
2,3,C003,Q003,Economists measure economic growth by construc...,NONE,"DOC1:p19,DOC1:p33",answered
3,4,C004,Q004,The world economy experienced a severe recessi...,NONE,"DOC1:p21,DOC1:p5",answered
4,5,C005,Q005,The U.S. unemployment rate reached close to 10...,NONE,"DOC1:p24,DOC2:p1",answered
5,6,C006,Q006,China's economy grew at an annual rate of 10.5...,NONE,DOC1:p13,answered
6,7,C007,Q007,The 2008 crisis tanked stock markets everywher...,Figure 1-1,"DOC1:p2,DOC1:Figure 1-1",answered
7,8,C008,Q008,"Yes, fast economic growth generally means fewe...",NONE,"DOC1:p24,DOC1:p29",answered
8,9,C009,Q009,"Yes, consumer prices and overall economic pric...",NONE,DOC1:p27,answered
9,10,C010,Q010,Europe has struggled with jobs even before the...,NONE,"DOC1:p12,DOC1:p9",answered


In [11]:
# Cell 8 - Task 1.4 evaluation on YOUR golden set (golden/golden_set.csv): baseline (dense only) vs final (hybrid).
import pandas as pd, pathlib
from rag.evaluate import evaluate
gpath = next((p for p in ["golden/golden_set.csv", "../golden/golden_set.csv"] if pathlib.Path(p).exists()), None)
golden = pd.read_csv(gpath, dtype=str, keep_default_na=False) if gpath else pd.DataFrame()
if len(golden):
    results = [evaluate(system, golden, mode=m)[0] for m in ("dense", "hybrid")]
    display(pd.DataFrame(results))
else:
    print("golden/golden_set.csv is empty - write your 25 questions first (see README).")

golden/golden_set.csv is empty - write your 25 questions first (see README).


In [12]:
import pandas as pd
s = pd.read_csv("submission.csv", dtype=str, keep_default_na=False)
print(s.status.value_counts())
print(s.shape)
s[["id","answer","status","figure_references","citations"]].head(12)

status
answered            38
not_found            1
session_override     1
Name: count, dtype: int64
(40, 7)


,id,answer,status,figure_references,citations
0,1,The global economic crisis around 2008 was spa...,answered,NONE,"DOC1:p2,DOC1:p5"
1,2,Unemployment rates should be of concern becaus...,answered,NONE,"DOC1:p24,DOC1:p30"
2,3,Economists measure economic growth by construc...,answered,NONE,"DOC1:p19,DOC1:p33"
3,4,The world economy experienced a severe recessi...,answered,NONE,"DOC1:p21,DOC1:p5"
4,5,The U.S. unemployment rate reached close to 10...,answered,NONE,"DOC1:p24,DOC2:p1"
5,6,China's economy grew at an annual rate of 10.5...,answered,NONE,DOC1:p13
6,7,The 2008 crisis tanked stock markets everywher...,answered,Figure 1-1,"DOC1:p2,DOC1:Figure 1-1"
7,8,"Yes, fast economic growth generally means fewe...",answered,NONE,"DOC1:p24,DOC1:p29"
8,9,"Yes, consumer prices and overall economic pric...",answered,NONE,DOC1:p27
9,10,Europe has struggled with jobs even before the...,answered,NONE,"DOC1:p12,DOC1:p9"


In [15]:
#Download it from Colab
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>